In [ ]:
# Reads singular provided YT Link
# Outputs pandas dataframe containing frame #, blue score, red score, timer, red location (left or right), blue team numbers, and red team numbers

import os
import re
import cv2
import pandas as pd
from yt_dlp import YoutubeDL
import imageio_ffmpeg
from ultralytics import YOLO
import easyocr


# --- User Variables --- 

YOUTUBE_LINK = "https://www.youtube.com/watch?v=NSWVoO4ZDEs" # video link to process
VIDEOS_PATH = "videos" # The directory where the downloaded videos are stored.

CROP_MODEL_PATH = "models/crop_scoreboard.pt" # The path of the YOLOv8 model that is used to crop the scoreboard from the raw videos.
INFO_MODEL_PATH = "models/extract_scoreboard_info.pt" # The path of the YOLOv8 model that extracts the information from the cropped scoreboard.

FRAME_SKIP = 15 # How many frames are skipped between each processing step. If the value is 15, it will process 1 frame every 15 frames.

DEVICE = 0  # This variable specifies what GPU(s) you use (if available). Can be set to "cpu", 0, [0,1], etc.

DELETE_VIDEO = False # Whether to delete the downloaded video after processing to save space. Set to True to enable deletion.

# Initialize directories and models
os.makedirs(VIDEOS_PATH, exist_ok=True)

crop_model = YOLO(CROP_MODEL_PATH)
info_model = YOLO(INFO_MODEL_PATH)

reader = easyocr.Reader(['en'], gpu=(DEVICE != "cpu"))

# Used to enhance the region of interest (ROI) for better OCR performance. It resizes, increases contrast, and applies thresholding.
def preprocess_for_ocr(roi):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    gray = cv2.convertScaleAbs(gray, alpha=1.8, beta=10)

    _, thresh = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    thresh_inv = cv2.bitwise_not(thresh)
    return [thresh, thresh_inv]

# Runs OCR on multiple images and returns the highest-confidence text
def read_best_text(images, allowlist):
    best_text, best_conf = None, 0

    for img in images:
        results = reader.readtext(
            img,
            allowlist=allowlist,
            detail=1,
            paragraph=False
        )
        for (_, text, conf) in results:
            if conf > best_conf:
                best_conf, best_text = conf, text

    return best_text

# Extracts the numeric value from the image using OCR
def read_number(img):
    images = preprocess_for_ocr(img)
    text = read_best_text(images, '0123456789')

    if text is None:
        return None

    text = re.sub(r"\D", "", text)
    return int(text) if text else None

# Extracts the timer value in MM:SS format from the image using OCR
def read_timer(img):
    images = preprocess_for_ocr(img)
    text = read_best_text(images, '0123456789:')

    if text is None:
        return None

    match = re.search(r"\d{1,2}:\d{2}", text)
    return match.group(0) if match else None

# Download video at best quality and return the saved filename
def download_video(url, output_dir):
    ydl_opts = {
        "format": "bestvideo+bestaudio/best",
        "outtmpl": os.path.join(output_dir, "%(title)s.%(ext)s"),
        "ffmpeg_location": imageio_ffmpeg.get_ffmpeg_exe(),
        "merge_output_format": "mp4",
    }

    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        filename = ydl.prepare_filename(info)
        return os.path.splitext(os.path.basename(filename))[0] + ".mp4"

# Download the video
video_filename = download_video(YOUTUBE_LINK, VIDEOS_PATH)
video_path = os.path.join(VIDEOS_PATH, video_filename)

# Storage for final rows and tracking previous scores
rows = []
prev_blue = None
prev_red = None
pending_row = None # Holds the first occurence of a score change when the timer is missing
is_auto = True

# Extract the youtube url for this video
youtube_url = YOUTUBE_LINK

cap = cv2.VideoCapture(video_path)
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Skip frames according to FRAME_SKIP
    if frame_idx % FRAME_SKIP != 0:
        frame_idx += 1
        continue
    
    # Detect the scoreboard region
    crop_results = crop_model(frame, device=DEVICE)[0]

    # Skip if no scoreboard is found
    if len(crop_results.boxes) == 0:
        frame_idx += 1
        continue

    # Crop scoreboard from frame
    x1, y1, x2, y2 = map(int, crop_results.boxes.xyxy[0])
    scoreboard = frame[y1:y2, x1:x2]

    # Find the elements inside the cropped frame (scores, timer, team numbers)
    info_results = info_model(scoreboard, device=DEVICE)[0]

    # Initialize variables for this frame
    blue_score = None
    red_score = None
    timer = None


    blue_center_x = None
    red_center_x = None

    team_data = []

    # Loop through each of the detected elements
    for b in info_results.boxes:

        cls_id = int(b.cls[0])
        label = info_model.names[cls_id]

        x1, y1, x2, y2 = map(int, b.xyxy[0])
        region = scoreboard[y1:y2, x1:x2]

        x_center = (x1 + x2) / 2

        # Read blue score
        if label == "blue_score":
            blue_score = read_number(region)
            blue_center_x = x_center

        # Read red score
        elif label == "red_score":
            red_score = read_number(region)
            red_center_x = x_center

        # Read timer
        elif label == "timer":
            timer = read_timer(region)

        # Read team number
        elif label == "team_number":
            num = read_number(region)
            if num is not None:
                team_data.append((num, x_center))

    # If both scores and timer are missing, skip the frame
    if blue_score is None and red_score is None and timer is None:
        frame_idx += 1
        continue

    # Ensure blue score doesn't decrease compared to previous frames
    if prev_blue is not None and blue_score is not None:
        if blue_score < prev_blue:
            frame_idx += 1
            continue

    # Ensure red score doesn't decrease compared to previous frames
    if prev_red is not None and red_score is not None:
        if red_score < prev_red:
            frame_idx += 1
            continue

    # If the scores haven't changed, skip the frame
    if prev_blue == blue_score and prev_red == red_score:
        frame_idx += 1
        continue

    # Determine if it is still auto stage or not
    if timer is not None and is_auto:
        try:
            minutes = int(timer.split(":")[0])
            if minutes == 2:
                is_auto = False
        except:
            pass

    # Identify if the red score is on the left or right side of the scoreboard
    red_location = None
    if blue_center_x is not None and red_center_x is not None:
        red_location = "right" if red_center_x > blue_center_x else "left"

    # Split the team numbers into blue vs red based on their loation to the score
    blue_teams = []
    red_teams = []

    midpoint = (blue_center_x + red_center_x) / 2 if blue_center_x and red_center_x else None

    if midpoint:
        for num, x in team_data:
            if red_location == "right":
                if x < midpoint:
                    blue_teams.append((num, abs(x - blue_center_x)))
                else:
                    red_teams.append((num, abs(x - red_center_x)))
            else:
                if x > midpoint:
                    blue_teams.append((num, abs(x - blue_center_x)))
                else:
                    red_teams.append((num, abs(x - red_center_x)))

    # Keep the closest 3 teams for each alliance
    blue_team_numbers = [n for n, _ in sorted(blue_teams, key=lambda x: x[1])[:3]]
    red_team_numbers = [n for n, _ in sorted(red_teams, key=lambda x: x[1])[:3]]

    # Build row
    current_row = {
        "Frame": frame_idx,
        "blue_score": blue_score,
        "red_score": red_score,
        "timer": timer,
        "is_auto": is_auto,
        "red_location": red_location,
        "blue_team_numbers": blue_team_numbers,
        "red_team_numbers": red_team_numbers,
        "youtube_link": youtube_url,
    }

    # If the timer is missing, store the very first instance for this score change
    if blue_score is not None and red_score is not None and timer is None:
        if pending_row is None:
            pending_row = current_row
        frame_idx += 1
        continue

    # If the timer ends up appearing in a later frame with the same scores, use that row instead
    if (
        pending_row is not None and
        timer is not None and
        blue_score == pending_row["blue_score"] and
        red_score == pending_row["red_score"]
    ):
        rows.append(current_row)
        pending_row = None

    # If score changes, flush the pending row and add the current row
    else:
        if pending_row is not None:
            rows.append(pending_row)
            pending_row = None

        rows.append(current_row)

    # Update previous scores
    prev_blue = blue_score
    prev_red = red_score

    frame_idx += 1

# Save any remaining pending row at the end of the video
if pending_row is not None:
    rows.append(pending_row)
    pending_row = None

cap.release()

# Delete video after processing to save space
if DELETE_VIDEO == True and os.path.exists(video_path):
    os.remove(video_path)

# Convert to dataframe and display
output_df = pd.DataFrame(rows)
output_df

[youtube] Extracting URL: https://www.youtube.com/watch?v=NSWVoO4ZDEs
[youtube] NSWVoO4ZDEs: Downloading webpage


[youtube] NSWVoO4ZDEs: Downloading android vr player API JSON
[info] NSWVoO4ZDEs: Downloading 1 format(s): 137+251
[download] Destination: videos/Final 1 - 2025 Rocket City Regional.f137.mp4
[download] 100% of   91.87MiB in 00:00:04 at 19.52MiB/s    
[download] Destination: videos/Final 1 - 2025 Rocket City Regional.f251.webm
[download] 100% of    2.30MiB in 00:00:00 at 16.54MiB/s  
[Merger] Merging formats into "videos/Final 1 - 2025 Rocket City Regional.mp4"
Deleting original file videos/Final 1 - 2025 Rocket City Regional.f137.mp4 (pass -k to keep)
Deleting original file videos/Final 1 - 2025 Rocket City Regional.f251.webm (pass -k to keep)

0: 384x640 (no detections), 7.2ms
Speed: 1.3ms preprocess, 7.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 7.0ms
Speed: 1.5ms preprocess, 7.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 6.9ms
Speed: 1.2ms preprocess, 6.9ms inference, 0.5ms pos

,Frame,blue_score,red_score,timer,red_location,blue_team_numbers,red_team_numbers,youtube_link
0,105,0,0,0:00,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
1,315,9,0,0:13,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
2,345,9,7,0:12,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
3,360,16,14,0:11,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
4,375,23,17,0:11,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
5,390,23,23,0:10,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
6,495,30,23,0:07,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
7,510,37,23,0:06,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
8,570,37,28,0:04,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs
9,585,37,37,0:04,right,"[10011, 4020, 2338]","[2783, 4028, 7111]",https://www.youtube.com/watch?v=NSWVoO4ZDEs


In [ ]:
# Reads from the CSV with multiple YT Links and processes each video

import os
import re
import cv2
import pandas as pd
from yt_dlp import YoutubeDL
import imageio_ffmpeg
from ultralytics import YOLO
import easyocr


# --- User Variables --- 

CSV_PATH = "frc_2025_matches.csv" # The path of the CSV file containing the youtube links
VIDEOS_PATH = "data_processing/videos" # The directory where the downloaded videos are stored.

CROP_MODEL_PATH = "models/crop_scoreboard.pt" # The path of the YOLOv8 model that is used to crop the scoreboard from the raw videos.
INFO_MODEL_PATH = "models/extract_scoreboard_info.pt" # The path of the YOLOv8 model that extracts the information from the cropped scoreboard.

DOWNLOAD_LIMIT = None # The max number of videos to download/process. Set to None to process all videos in the CSV.
FRAME_SKIP = 15 # How many frames are skipped between each processing step. If the value is 15, it will process 1 frame every 15 frames.

DEVICE = 0  # This variable specifies what GPU(s) you use (if available). Can be set to "cpu", 0, [0,1], etc.

# --- INITIALIZE MODELS ---

os.makedirs(VIDEOS_PATH, exist_ok=True)

crop_model = YOLO(CROP_MODEL_PATH)
info_model = YOLO(INFO_MODEL_PATH)

reader = easyocr.Reader(['en'], gpu=(DEVICE != "cpu"))

# --- OCR FUNCTIONS ---

# Used to enhance the region of interest (ROI) for better OCR performance. It resizes, increases contrast, and applies thresholding.
def preprocess_for_ocr(roi):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    gray = cv2.convertScaleAbs(gray, alpha=1.8, beta=10)

    _, thresh = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    thresh_inv = cv2.bitwise_not(thresh)
    return [thresh, thresh_inv]

# Runs OCR on multiple images and returns the highest-confidence text
def read_best_text(images, allowlist):
    best_text, best_conf = None, 0

    for img in images:
        results = reader.readtext(
            img,
            allowlist=allowlist,
            detail=1,
            paragraph=False
        )
        for (_, text, conf) in results:
            if conf > best_conf:
                best_conf, best_text = conf, text

    return best_text

# Extracts the numeric value from the image using OCR
def read_number(img):
    images = preprocess_for_ocr(img)
    text = read_best_text(images, '0123456789')

    if text is None:
        return None

    text = re.sub(r"\D", "", text)
    return int(text) if text else None

# Extracts the timer value in MM:SS format from the image using OCR
def read_timer(img):
    images = preprocess_for_ocr(img)
    text = read_best_text(images, '0123456789:')

    if text is None:
        return None

    match = re.search(r"\d{1,2}:\d{2}", text)
    return match.group(0) if match else None

# Download video at best quality and return the saved filename
def download_video(url, output_dir):
    ydl_opts = {
        "format": "bestvideo+bestaudio/best",
        "outtmpl": os.path.join(output_dir, "%(title)s.%(ext)s"),
        "ffmpeg_location": imageio_ffmpeg.get_ffmpeg_exe(),
        "merge_output_format": "mp4",
    }

    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        filename = ydl.prepare_filename(info)
        return os.path.splitext(os.path.basename(filename))[0] + ".mp4"

# Load the match CSV
df = pd.read_csv(CSV_PATH)

# Track download count and video file paths
download_count = 0
video_files = []

# Map existing video files to avoid re-downloading
existing_files = {
    f: os.path.join(VIDEOS_PATH, f)
    for f in os.listdir(VIDEOS_PATH)
    if f.lower().endswith(".mp4")
}

# Iterate through matches and download videos if not already present, respecting the download limit
for _, row in df.iterrows():

    url = row["youtube_link"]

    # Use existing file if already downloaded
    if len(existing_files) > 0:
        video_files.extend(existing_files.values())
        break
    
    # Download videos up to the specified DOWNLOAD_LIMIT
    if DOWNLOAD_LIMIT is None or download_count < DOWNLOAD_LIMIT:
        filename = download_video(url, VIDEOS_PATH)
        video_files.append(os.path.join(VIDEOS_PATH, filename))
        download_count += 1

# Storage for final rows and tracking previous scores
rows = []
prev_blue = None
prev_red = None
pending_row = None # Holds the first occurence of a score change when the timer is missing

# Process each video
for video_path in video_files:

    # Extract the youtube url for this video
    youtube_url = df.iloc[video_files.index(video_path)]["youtube_link"]

    cap = cv2.VideoCapture(video_path)
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Skip frames according to FRAME_SKIP
        if frame_idx % FRAME_SKIP != 0:
            frame_idx += 1
            continue
        
        # Detect the scoreboard region
        crop_results = crop_model(frame, device=DEVICE)[0]

        # Skip if no scoreboard is found
        if len(crop_results.boxes) == 0:
            frame_idx += 1
            continue

        # Crop scoreboard from frame
        x1, y1, x2, y2 = map(int, crop_results.boxes.xyxy[0])
        scoreboard = frame[y1:y2, x1:x2]

        # Find the elements inside the cropped frame (scores, timer, team numbers)
        info_results = info_model(scoreboard, device=DEVICE)[0]

        # Initialize variables for this frame
        blue_score = None
        red_score = None
        timer = None

        blue_center_x = None
        red_center_x = None

        team_data = []

        # Loop through each of the detected elements
        for b in info_results.boxes:

            cls_id = int(b.cls[0])
            label = info_model.names[cls_id]

            x1, y1, x2, y2 = map(int, b.xyxy[0])
            region = scoreboard[y1:y2, x1:x2]

            x_center = (x1 + x2) / 2

            # Read blue score
            if label == "blue_score":
                blue_score = read_number(region)
                blue_center_x = x_center

            # Read red score
            elif label == "red_score":
                red_score = read_number(region)
                red_center_x = x_center

            # Read timer
            elif label == "timer":
                timer = read_timer(region)

            # Read team number
            elif label == "team_number":
                num = read_number(region)
                if num is not None:
                    team_data.append((num, x_center))

        # If both scores and timer are missing, skip the frame
        if blue_score is None and red_score is None and timer is None:
            frame_idx += 1
            continue

        # Ensure blue score doesn't decrease compared to previous frames
        if prev_blue is not None and blue_score is not None:
            if blue_score < prev_blue:
                frame_idx += 1
                continue

        # Ensure red score doesn't decrease compared to previous frames
        if prev_red is not None and red_score is not None:
            if red_score < prev_red:
                frame_idx += 1
                continue

        # If the scores haven't changed, skip the frame
        if prev_blue == blue_score and prev_red == red_score:
            frame_idx += 1
            continue

        # Identify if the red score is on the left or right side of the scoreboard
        red_location = None
        if blue_center_x is not None and red_center_x is not None:
            red_location = "right" if red_center_x > blue_center_x else "left"

        # Split the team numbers into blue vs red based on their loation to the score
        blue_teams = []
        red_teams = []

        midpoint = (blue_center_x + red_center_x) / 2 if blue_center_x and red_center_x else None

        if midpoint:
            for num, x in team_data:
                if red_location == "right":
                    if x < midpoint:
                        blue_teams.append((num, abs(x - blue_center_x)))
                    else:
                        red_teams.append((num, abs(x - red_center_x)))
                else:
                    if x > midpoint:
                        blue_teams.append((num, abs(x - blue_center_x)))
                    else:
                        red_teams.append((num, abs(x - red_center_x)))

        # Keep the closest 3 teams for each alliance
        blue_team_numbers = [n for n, _ in sorted(blue_teams, key=lambda x: x[1])[:3]]
        red_team_numbers = [n for n, _ in sorted(red_teams, key=lambda x: x[1])[:3]]

        # Build row
        current_row = {
            "Frame": frame_idx,
            "blue_score": blue_score,
            "red_score": red_score,
            "timer": timer,
            "red_location": red_location,
            "blue_team_numbers": blue_team_numbers,
            "red_team_numbers": red_team_numbers,
            "youtube_link": youtube_url,
        }

        # If the timer is missing, store the very first instance for this score change
        if blue_score is not None and red_score is not None and timer is None:
            if pending_row is None:
                pending_row = current_row
            frame_idx += 1
            continue

        # If the timer ends up appearing in a later frame with the same scores, use that row instead
        if (
            pending_row is not None and
            timer is not None and
            blue_score == pending_row["blue_score"] and
            red_score == pending_row["red_score"]
        ):
            rows.append(current_row)
            pending_row = None

        # If score changes, flush the pending row and add the current row
        else:
            if pending_row is not None:
                rows.append(pending_row)
                pending_row = None

            rows.append(current_row)

        # Update previous scores
        prev_blue = blue_score
        prev_red = red_score

        frame_idx += 1

    # Save any remaining pending row at the end of the video
    if pending_row is not None:
        rows.append(pending_row)
        pending_row = None

    cap.release()

# Convert to dataframe and display
output_df = pd.DataFrame(rows)
output_df